# Combine data: REDS Index and REDS Recall
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from hk1_r12ter_analysis.config import (
    GENE_ID,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    logger,
)
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Load REDS data

In [ ]:
normalization_inputs = (
    # Sample Normalization, Data Transformation, and Data Scaling
    [None, None, None],
    ["median", None, "auto"],
    ["median", "log2", "pareto"],  # For limma package
)

input_dirpath = INTERIM_DATA_DIR / "REDS"
output_dirpath = PROCESSED_DATA_DIR / "REDS"
output_dirpath.mkdir(parents=True, exist_ok=True)

### Load REDS Index data

In [ ]:
sample_key = "INDEX DONOR ID"
group_key = None
source_type = "REDS-Index"

#### Combine metadata and genomics

In [ ]:
data_types = [f"Genomics-{GENE_ID}", "Metadata"]
combination_data_type = "_".join(data_types)
filename_args = (source_type, "Donor", combination_data_type)
logger.debug(f"Procesing data for '{'-'.join(filename_args)}'...")
logger.debug(f"Combining data for '{'-'.join(filename_args)}'...")
for idx, data_type in enumerate(data_types):
    logger.debug(f"Adding '{source_type}-{data_type}' data...")
    filename = make_filename(source_type, "Donor", data_type)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=sample_key)
    df.columns = pd.MultiIndex.from_tuples([(data_type, col) for col in df.columns])
    df_merged = df_merged.join(df, how="outer") if idx != 0 else df
    logger.info(f"Added '{source_type}-{data_type}' data.")
# Remove duplicates, keeping only those added last
logger.info(f"Combined data for '{source_type}-{combination_data_type}'.")
#  Save combined data
filename = make_filename(source_type, "Donor", combination_data_type)
save_dataframe_to_file(df_merged, output_dirpath / filename, index=True)
logger.info(f"Processed data for '{source_type}-{combination_data_type}'.")
df_merged

#### Combine normalized data with genomics and metadata
##### Metadata + Genomics + Metabolomics + Proteomics

In [ ]:
combinations_data_types = {
    "All_Data": ["Metadata", f"Genomics-{GENE_ID}", "Proteomics", "Metabolomics"],
}
normalization_strs = [
    "-".join([str(x).lower() for x in norm_inputs]) for norm_inputs in normalization_inputs
]
index_cols = [key for key in [sample_key, group_key] if key]
for norm_str in normalization_strs:
    # Set directory paths
    input_dirpath_norm = input_dirpath / "Normalized" / norm_str
    output_dirpath_norm = output_dirpath / "Normalized" / norm_str
    output_dirpath_norm.mkdir(parents=True, exist_ok=True)
    logger.debug(
        "Combining data using sample normalization: {}; data transformation: {}; data scaling: {}.".format(
            *norm_str.split("-")
        )
    )

    # Iterate through combinations
    for combination_data_type, data_types in combinations_data_types.items():
        logger.debug(f"Procesing data for '{source_type}-{combination_data_type}'...")
        # Load metadata and genomics
        logger.debug(f"Combining data for '{source_type}-{combination_data_type}'...")
        for idx, data_type in enumerate(data_types):
            logger.debug(f"Adding '{source_type}-{data_type}' data...")
            filename = make_filename(source_type, "Donor", "RBC", data_type)
            if (input_dirpath_norm / filename).exists():
                df = load_dataframe_from_file(input_dirpath_norm / filename, index_col=index_cols)
            elif (input_dirpath / filename).exists():
                df = load_dataframe_from_file(input_dirpath / filename, index_col=index_cols)
            else:
                filename = make_filename(source_type, "Donor", data_type)
                df = load_dataframe_from_file(input_dirpath / filename, index_col=sample_key)
            df.columns = pd.MultiIndex.from_tuples([(data_type, col) for col in df.columns])
            df_merged = df_merged.join(df, how="outer") if idx != 0 else df
            logger.info(f"Added '{source_type}-{data_type}' data.")
        # Remove duplicates, keeping only those added last
        logger.info(f"Combined data for '{source_type}-{combination_data_type}'.")
        #  Save combined data
        filename = make_filename(source_type, "Donor", "RBC", combination_data_type)
        save_dataframe_to_file(df_merged, output_dirpath_norm / filename, index=True)
        logger.info(f"Processed data for '{source_type}-{combination_data_type}'.")
    logger.info(
        "Combined data for sample normalization: {}; data transformation: {}; data scaling: {}.".format(
            *norm_str.split("-")
        )
    )

### Load REDS Recall data

In [ ]:
sample_key = "RECALL DONOR ID"
group_key = "Day"
source_type = "REDS-Recall"

#### Combine metadata and genomics

In [ ]:
data_types = [f"Genomics-{GENE_ID}", "Metadata"]
combination_data_type = "_".join(data_types)
filename_args = (source_type, "Donor", combination_data_type)
logger.debug(f"Procesing data for '{'-'.join(filename_args)}'...")
logger.debug(f"Combining data for '{'-'.join(filename_args)}'...")
for idx, data_type in enumerate(data_types):
    logger.debug(f"Adding '{source_type}-{data_type}' data...")
    filename = make_filename(source_type, "Donor", data_type)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=sample_key)
    df.columns = pd.MultiIndex.from_tuples([(data_type, col) for col in df.columns])
    df_merged = df_merged.join(df, how="outer") if idx != 0 else df
    logger.info(f"Added '{source_type}-{data_type}' data.")
# Remove duplicates, keeping only those added last
logger.info(f"Combined data for '{source_type}-{combination_data_type}'.")
#  Save combined data
filename = make_filename(source_type, "Donor", combination_data_type)
save_dataframe_to_file(df_merged, output_dirpath / filename, index=True)
logger.info(f"Processed data for '{source_type}-{combination_data_type}'.")
df_merged

#### Combine normalized data with genomics and metadata
##### Metadata + Genomics + Metabolomics + Proteomics + Lipidomics + Trace Elements

In [ ]:
combinations_data_types = {
    "All_Data": [
        "Metadata",
        f"Genomics-{GENE_ID}",
        "Proteomics",
        "Metabolomics",
        "Lipidomics",
        # "Lipidomics-DegUnsat",
        # "Lipidomics-LipidClass",
        "TraceElements",
        # "Vesicles-mtDNA",
    ],
}
normalization_strs = [
    "-".join([str(x).lower() for x in norm_inputs]) for norm_inputs in normalization_inputs
]
index_cols = [key for key in [sample_key, group_key] if key]
for norm_str in normalization_strs:
    # Set directory paths
    input_dirpath_norm = input_dirpath / "Normalized" / norm_str
    output_dirpath_norm = output_dirpath / "Normalized" / norm_str
    output_dirpath_norm.mkdir(parents=True, exist_ok=True)
    logger.debug(
        "Combining data using sample normalization: {}; data transformation: {}; data scaling: {}.".format(
            *norm_str.split("-")
        )
    )

    # Iterate through combinations
    for combination_data_type, data_types in combinations_data_types.items():
        logger.debug(f"Procesing data for '{source_type}-{combination_data_type}'...")
        # Load metadata and genomics
        logger.debug(f"Combining data for '{source_type}-{combination_data_type}'...")
        for idx, data_type in enumerate(data_types):
            logger.debug(f"Adding '{source_type}-{data_type}' data...")
            filename = make_filename(source_type, "Donor", "RBC", data_type)
            if (input_dirpath_norm / filename).exists():
                df = load_dataframe_from_file(input_dirpath_norm / filename, index_col=index_cols)
            elif (input_dirpath / filename).exists():
                df = load_dataframe_from_file(input_dirpath / filename, index_col=index_cols)
            else:
                filename = make_filename(source_type, "Donor", data_type)
                df = load_dataframe_from_file(input_dirpath / filename, index_col=sample_key)
            df.columns = pd.MultiIndex.from_tuples([(data_type, col) for col in df.columns])
            df_merged = df_merged.join(df, how="outer") if idx != 0 else df
            logger.info(f"Added '{source_type}-{data_type}' data.")
        # Remove duplicates, keeping only those added last
        logger.info(f"Combined data for '{source_type}-{combination_data_type}'.")
        #  Save combined data
        filename = make_filename(source_type, "Donor", "RBC", combination_data_type)
        save_dataframe_to_file(df_merged, output_dirpath_norm / filename, index=True)
        logger.info(f"Processed data for '{source_type}-{combination_data_type}'.")
    logger.info(
        "Combined data for sample normalization: {}; data transformation: {}; data scaling: {}.".format(
            *norm_str.split("-")
        )
    )